<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/ct_concept_steering_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Concept-Direction Steering Demo

Demonstrates interpretune's **concept-direction-mediated, sign-aware, multi-feature steering** on a
trivial example, `orange` color-vs-fruit sense disambiguation to familiarize the user with some of interpretune's
intervention mechanisms:

1. **Feature-mediated path**: concept direction -> attribution graph -> sign-aware
   `FeatureSelectionSpec` top-feature selection -> `feature_intervention_forward` (circuit-tracer
   feature interventions with sign-aware, influence-normalized scaling).
2. **Direct-hook path**: the same concept direction applied via `model_fwd_intervention`
   (hook-tensor add/project interventions at canonical hook points).

Both paths derive the concept direction from the token-embedding basis (`paired_rejection`
over the concept groups).

The `BACKEND` parameter selects the circuit-tracer backend for all steps: NNsight (default)
or TransformerLens — both are validated by the parameterized notebook tests. (The
TransformerLens backend uses the legacy `HookedTransformer` path; circuit-tracer does not yet
support `TransformerBridge` — see the tracking notes in `docs/circuit_tracer_backend_support.md`.)

This notebook runs `gemma-2-2b` + the Gemma Scope `gemmascope-transcoder-16k` set against the
public [neuronpedia.org](https://www.neuronpedia.org) dashboards, so every selected feature's
semantics can be inspected by clicking through — no local services required. That matches the
substrate guidance in `tests/nb_experiments/EXPERIMENT_STATUS.md` (base model + base-trained
transcoders).

Running against a local Neuronpedia dev webapp instead — with locally generated feature
explanations and the instruction-tuned `gemma-3-1b-it` substrate — is a separate notebook:
[`ct_concept_steering_demo_local_np.ipynb`](ct_concept_steering_demo_local_np.ipynb).

> **Prerequisites**: a GPU with bf16 support, plus access to the model and transcoder weights.
> Nothing else — dashboard links and explanations come from the public Neuronpedia API.

> **Expected result**: the two steering paths are not equally strong. The attribution-graph
> feature-mediated path (step 2) is the one selecting features *for their causal effect on the target
> logit difference*, and it should produce the largest target-gap shift. Direct-hook steering (step 4)
> applies a concept direction at a hook point without that per-feature attribution, so it is expected
> to be weaker. That gap is a finding, not a defect: step 5's decoupling analysis exists to explain it.


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
BACKEND = "nnsight"  # circuit-tracer backend for all steps: "nnsight" or "transformerlens"
CONCEPT_PROMPT = "Is orange a color or a fruit? Answer with one word: Color or Fruit. orange ->"
CONCEPT_TARGET_TOKENS = ["Fruit", "Color"]
FEATURE_SELECTION_TOP_N = 5
FEATURE_SELECTION_MIN_LAYER = 10  # fs_l10_n5 lineage: layers >= 10
FEATURE_SELECTION_SCORE_SIGN = "any"  # any | positive | negative
INTERVENTION_SCALE_FACTOR = 20.0  # validated s5_any demo scale
EMBED_INTERVENTION_MODE = "add"  # model_fwd_intervention mode for the embed step
EMBED_INTERVENTION_HOOK = "unembed.hook_in"

# -- Model / dashboard substrate: public gemma-2-2b + public neuronpedia.org dashboards ---------
# Base model + base-trained transcoders, so every selected feature can be inspected directly on
# neuronpedia.org. The local-Neuronpedia substrate lives in ct_concept_steering_demo_local_np.ipynb.
REGISTRY_KEY = "rte_demo.gemma2.circuit_tracer"  # hub component configuration key (model + backend)
MODEL_NAME = "gemma-2-2b"
TRANSCODER_SET = "gemma"  # circuit-tracer transcoder set override (None keeps the registry default)
NEURONPEDIA_MODEL_ID = "gemma-2-2b"
NEURONPEDIA_SOURCE_SET = "gemmascope-transcoder-16k"
CHAT_FORMAT_PROMPT = False  # True for instruction-tuned models (render CONCEPT_PROMPT via chat template)

# -- Section 4b (J-space patch steering) --------------------------------------------------------
JLENS_OPS_REPO = "speediedan/jlens_steering_ops"  # private until #261 flips it public; needs HF auth with access
JLENS_OPS_REVISION = "872760c7caa4a503a8b12ed21960dd46a08786df"  # pinned: trusted hub code must not change under us
JLENS_LAYER = 24  # fitted lens layer; later layers suit naming (gemma-2-2b flips at 24, the last fitted)
JLENS_PATCH_SCALE = 1.0  # a pure coordinate swap -- the J-space path needs no oversteering
RUN_JSPACE_SECTION = True  # skips cleanly (with the reason) when the collection is unreachable

In [3]:
# @title Imports { display-mode: "form" }
import torch  # noqa: F401

import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis.backends import FeatureSelectionSpec  # noqa: F401
from it_examples.utils.nb_ui_utils import (  # noqa: F401
    best_variant_token_ids,
    display_steering_results,
    display_target_gap,
    display_top_features_comparison,
    resolve_feature_explanations,
)

## 1. Session setup

Single-backend circuit-tracer session built from the `REGISTRY_KEY` hub component configuration
(default: `rte_demo.gemma2.circuit_tracer` with the Gemma Scope transcoder set that matches the
public `gemma-2-2b` dashboards).

> **Note:** resolution is cache-backed and per-key — `it.hub.load` hydrates only the requested
> configuration (`ensure_local_seeds` materializes the in-tree seed components into the local
> components cache first; both calls are offline and idempotent).


In [4]:
# @title 1: Session construction { display-mode: "form" }
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it
from it_examples.seeds import ensure_local_seeds
from interpretune import ITSession, ITSessionConfig

# load HF credentials before session init (model + transcoder downloads)
for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", REGISTRY_KEY)
# single circuit-tracer backend for all phases (BACKEND selects the replacement-model implementation)
base_it_cfg.circuit_tracer_cfg.backend = BACKEND
if TRANSCODER_SET:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = TRANSCODER_SET
if BACKEND == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    # the TL circuit-tracer backend needs the transformer_lens adapter in the composition
    # (it provides the replacement-model init path; see docs/circuit_tracer_backend_support.md)
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)
session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
tokenizer = module.replacement_model.tokenizer
print(f"session ready: {type(module).__name__} ({MODEL_NAME} + circuit-tracer {BACKEND} backend)")

~/.claude/jobs/b395eb70/tmp/wt594/src/interpretune/config/shared.py:325: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'tl_cfg': ITLensFromPretrainedNoProcessingConfig(move_to_device=True, default_padding_side='left', use_bridge=False, model_name='gemma-2-2b', fold_ln=False, center_writing_weights=False, center_unembed=False, refactor_factored_attn_matrices=False, checkpoint_index=None, checkpoint_value=None, hf_model=None, device='cuda', n_devices=1, tokenizer=None, fold_value_biases=False, default_prepend_bos=True, dtype='float32'), 'circuit_tracer_cfg': CircuitTracerConfig(backend='transformerlens', model_name=None, transcoder_set='gemma', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', lazy_encoder=None, lazy_decoder=True, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=Non

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule


[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.


session ready: InterpretunableModule (gemma-2-2b + circuit-tracer nnsight backend)


## 2. Feature-mediated path: concept direction -> attribution -> sign-aware selection -> feature steering

Runs the registered composite `it.intervention_from_concept(...)` (concept_direction ->
compute_attribution_graph -> graph_node_influence -> extract_top_features ->
feature_intervention_forward) with:

- `FeatureSelectionSpec(layer_slice=(FEATURE_SELECTION_MIN_LAYER, None), score_sign=FEATURE_SELECTION_SCORE_SIGN,
  score_source="signed_influence")`
- sign-aware, influence-normalized scaling
  (`intervention_sign_aware_scale=True`, `intervention_max_influence_norm_scale=True`,
  `intervention_value_source="top_feature_activation_values"`, `intervention_scale_factor=INTERVENTION_SCALE_FACTOR`)

Expected outcome: post-intervention target gap exceeds the pre-intervention gap and the
post-intervention argmax lands in the target-token variant set.


In [5]:
# @title 2: Feature-mediated steering { display-mode: "form" }
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.config import AnalysisCfg, init_analysis_cfgs

module.analysis_cfg = AnalysisCfg(target_op=it.compute_attribution_graph, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

fruits = ["apple", "banana", "grape", "peach"]
colors = ["red", "blue", "green", "yellow"]
if CHAT_FORMAT_PROMPT:
    # instruction-tuned replacement models assert chat-formatted inputs
    from it_examples.examples.prompt_configs.prompt_configs import GemmaPromptConfig

    prompt = GemmaPromptConfig().apply_chat_template_fn(
        tokenizer, CONCEPT_PROMPT, tokenize=False, add_generation_prompt=True
    )
else:
    prompt = CONCEPT_PROMPT

# sign-aware, influence-normalized scaling (the validated s5_any lineage)
ct_cfg = module.it_cfg.circuit_tracer_cfg
ct_cfg.intervention_sign_aware_scale = True
ct_cfg.intervention_max_influence_norm_scale = True
ct_cfg.intervention_value_source = "top_feature_activation_values"

selection_spec = FeatureSelectionSpec(
    layer_slice=slice(FEATURE_SELECTION_MIN_LAYER, None),
    score_source="signed_influence",
    score_sign=FEATURE_SELECTION_SCORE_SIGN,
    rank_by_abs=True,
)
pipeline_results = it.intervention_from_concept(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=FEATURE_SELECTION_TOP_N,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
    feature_selection=selection_spec,
)
# one call renders the linked/signed features table + the consolidated target-gap table and
# returns everything later phases need (features, direction, target ids, gaps)
DASHBOARD_BASE_URL = "https://www.neuronpedia.org"
steering_base_url = DASHBOARD_BASE_URL
steering = display_steering_results(
    pipeline_results,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
    neuronpedia_base_url=steering_base_url,
    min_layer=FEATURE_SELECTION_MIN_LAYER,
)
steered_features = steering.steered_features
pipeline_direction = steering.direction
target_a_id, target_b_id = steering.target_ids
fm_pre_gap, fm_post_gap = steering.pre_gap, steering.post_gap
assert fm_post_gap > fm_pre_gap, "feature-mediated steering should push the gap toward the target concept"

Phase 0: Precomputing activations and vectors


Precomputation completed in 2.03s


Found 18348 active features


Phase 1: Running forward pass


Forward pass completed in 0.55s


...

Feature influence computation: 100%|██████████| 8192/8192 [00:04<00:00, 1905.41it/s]


Feature attributions completed in 4.30s


Attribution completed in 10.31s


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,2.11e-08
2,"(24, 19, 13277)",−,1.97e-08
3,"(24, 19, 5999)",+,1.08e-08
4,"(24, 19, 3865)",+,5.31e-09
5,"(25, 19, 13210)",+,5.17e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.868%,25.1250,29.3750,+4.2500
Color,15.106%,1.03e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


## 3. Feature semantics: the top-features table

The steered features render as a table with the `(layer, pos, feature)` node tuple linked to its
dashboard, the **signed** influence score (Sign / |Score| columns — the `signed_influence`
selection can steer with negative-signed features, so the sign is colour-coded), and a best-effort
**Explanation** column resolved from the public Neuronpedia feature API.


In [6]:
# @title 3: Top-features table { display-mode: "form" }
# top_feature_ids are (layer, position, feature) tuples; dashboards/explanations are per
# (layer, feature), so collapse positions while preserving selection order
steered_layer_feature_pairs = list(dict.fromkeys((f[0], f[-1]) for f in steered_features))

# Best-effort explanation text from the public neuronpedia.org feature API; unmapped features
# simply render an empty Explanation cell
feature_explanations = resolve_feature_explanations(
    model_id=NEURONPEDIA_MODEL_ID,
    source_set=NEURONPEDIA_SOURCE_SET,
    feature_tuples=steered_layer_feature_pairs,
    base_url=DASHBOARD_BASE_URL,
)

display_top_features_comparison(
    {"Steered Features (signed influence)": steered_features},
    {"Steered Features (signed influence)": pipeline_results.top_feature_scores.tolist()},
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
    neuronpedia_base_url=DASHBOARD_BASE_URL,
    show_score_sign=True,
    feature_explanations=feature_explanations,
)

#,Node,Sign,|Score|,Explanation
1,"(25, 19, 16131)",−,2.11e-08,"the comparison of two varieties of fruit, relating to size, taste, color, and genetic information"
2,"(24, 19, 13277)",−,1.97e-08,words related to questions and requests
3,"(24, 19, 5999)",+,1.08e-08,"language related to institutions, negative situations, the internet, and programming languages"
4,"(24, 19, 3865)",+,5.31e-09,"dollar signs and other currency symbols, potentially alongside numbers or related terms like ""terms"" and ""bonus""."
5,"(25, 19, 13210)",+,5.17e-09,grammatical structures and parts of speech like noun phrases and verb phrases


## 4. Direct-hook path: concept direction -> hook-tensor steering

Recomputes the embed-basis concept direction for the same concept pair (a consistency check against
the step 2 pipeline's direction — cosine should be ~1.0 since both derive from the same
embedding-basis `paired_rejection`) and applies `it.model_fwd_intervention(...)` at
`EMBED_INTERVENTION_HOOK` in `EMBED_INTERVENTION_MODE` mode, comparing
`pre/post_intervention_logits` and the target-token gap against the feature-mediated result. The
same direction steered through selected transcoder features vs added directly at the hook point
produces different effect sizes — the feature-mediated path is typically stronger per unit scale.


In [7]:
# @title 4: Direct-hook steering { display-mode: "form" }
# Recompute the embed-basis concept direction for the same concept pair (no store rows -> embed basis)
direct_result = it.concept_direction(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color (direct)",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
    ),
    None,
    0,
)
direct_direction = direct_result.concept_direction.detach()
cosine = torch.nn.functional.cosine_similarity(
    pipeline_direction, direct_direction.float().cpu().reshape(-1), dim=0
).item()
print(f"pipeline-vs-direct direction cosine: {cosine:+.4f} (~1.0 expected — same embed-basis construction)")

# Direct hook-tensor intervention at the canonical hook point
module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

# chat-rendered prompts already carry their special tokens; plain completion prompts need them added
enc = tokenizer(prompt, return_tensors="pt", padding=False, add_special_tokens=not CHAT_FORMAT_PROMPT)
device = next(module.model.parameters()).device
if BACKEND == "transformerlens":
    # HookedTransformer.forward takes `input`, not the HF-style `input_ids`/`attention_mask` keys
    batch = {"input": enc["input_ids"].to(device)}
else:
    batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in dict(enc).items()}

# legacy HookedTransformer models (the CT TransformerLens backend) expose no `unembed.hook_in`;
# their pre-unembed equivalent is `ln_final.hook_normalized` (alias-map expansion tracked in
# interpretune#223)
intervention_hook = EMBED_INTERVENTION_HOOK
if BACKEND == "transformerlens" and EMBED_INTERVENTION_HOOK == "unembed.hook_in":
    intervention_hook = "ln_final.hook_normalized"

direct_batch = AnalysisBatch(
    prompts=[prompt],
    concept_direction=direct_direction,
    logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
    concept_group_a_token_ids=[target_a_id],
    concept_group_b_token_ids=[target_b_id],
    concept_cache_key=intervention_hook,
    intervention_hook_pattern=intervention_hook,
    intervention_mode=EMBED_INTERVENTION_MODE,
    direction_scale_factor=INTERVENTION_SCALE_FACTOR,
)
direct_out = it.model_fwd_intervention(module, direct_batch, batch, 0)

direct_pre_gap, direct_post_gap = display_target_gap(
    direct_out.pre_intervention_logits.float().cpu().reshape(-1),
    direct_out.post_intervention_logits.float().cpu().reshape(-1),
    (CONCEPT_TARGET_TOKENS[0], target_a_id),
    (CONCEPT_TARGET_TOKENS[1], target_b_id),
    title="Direct-hook steering — target gap",
)
print(
    f"feature-mediated delta {fm_post_gap - fm_pre_gap:+.3f} vs direct-hook delta "
    f"{direct_post_gap - direct_pre_gap:+.3f}"
)
assert direct_post_gap > direct_pre_gap, "direct-hook steering should push the gap toward the target concept"

pipeline-vs-direct direction cosine: +1.0000 (~1.0 expected — same embed-basis construction)


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,2.036%,25.1250,27.8750,+2.7500
Color,15.106%,1.797%,27.0000,27.7500,+0.7500
Gap (Fruit − Color),,,-1.8750,+0.1250,+2.0000


feature-mediated delta +9.375 vs direct-hook delta +2.000


## 4b. J-space path: workspace patch steering (hub op collection)

Steers the same concept pair a third way, in **J-space** (Gurnee et al. 2026, the [workspace
paper](https://transformer-circuits.pub/2026/workspace/)), using a **genuinely non-bundled op
collection** pulled from the Hub at run time: `JLENS_OPS_REPO` builds the `(2, d_model)` J-lens pole
pair (`v_c = (W_U[c] * final_norm_scale) @ J_l`, lenses pre-fitted in
[`neuronpedia/jacobian-lens`](https://huggingface.co/neuronpedia/jacobian-lens)) and stages a
lens-coordinate `patch` intervention, `h <- h + V(sigma(c) - c)` -- a coordinate **swap** that leaves
everything orthogonal to the pair untouched.

Contrast with the two paths above at matched displacement, not matched scale (`#339`: scale
conventions are not comparable across modes). On gemma-2-2b, the embed-basis `add` at scale 20
displaces 20 units and moves the gap by +20.4; the J-space swap at scale 1.0 displaces 154 units
and moves it by +5.7; the `add` at the swap's displacement (scale 154) moves it by +62.9. The
swap's virtue is surgicality -- a coordinate exchange that leaves everything orthogonal to the
pair untouched -- not efficiency: per unit displaced, the push steers an order of magnitude harder
and oversteers long before the swap flips.

The collection executes code from the Hub, so the cell opts into `IT_TRUST_REMOTE_CODE` and pins the
collection revision (see the [hub trust posture](https://interpretune.org/en/latest/usage/hub_trust_posture.html)).
It skips cleanly, with the reason, when the repo is unreachable (it is private until interpretune#261).


In [8]:
# @title 4b: J-space patch steering { display-mode: "form" }
import os

jspace_ran, jspace_skip_reason = False, "RUN_JSPACE_SECTION=False"
if RUN_JSPACE_SECTION:
    # Hub-resident op code: opt in explicitly and pin the revision so trusted code cannot change under us.
    os.environ.setdefault("IT_TRUST_REMOTE_CODE", "1")
    try:
        _, jlens_ops_commit = it.hub.pull_ops(JLENS_OPS_REPO, revision=JLENS_OPS_REVISION)
        jspace_ran = True
    except Exception as pull_err:  # private until #261 flips it public: no access/offline -> skip with the reason
        jspace_skip_reason = f"could not pull {JLENS_OPS_REPO}: {pull_err}"

if jspace_ran:
    print(f"pulled {JLENS_OPS_REPO} @ {jlens_ops_commit[:10]}")
    module.analysis_cfg = AnalysisCfg(target_op=it.jlens_patch_intervention, ignore_manual=True, save_tokens=False)
    init_analysis_cfgs(module, [module.analysis_cfg])
    jlens_batch = AnalysisBatch(
        prompts=[prompt],
        concept_group_a_token_ids=[target_a_id],
        concept_group_b_token_ids=[target_b_id],
        logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
        jlens_layer=JLENS_LAYER,
        direction_scale_factor=JLENS_PATCH_SCALE,
    )
    jlens_out = it.jlens_patch_intervention(module, jlens_batch, batch, 0)
    jl_pre_gap, jl_post_gap = display_target_gap(
        jlens_out.pre_intervention_logits.float().cpu().reshape(-1),
        jlens_out.post_intervention_logits.float().cpu().reshape(-1),
        (CONCEPT_TARGET_TOKENS[0], target_a_id),
        (CONCEPT_TARGET_TOKENS[1], target_b_id),
        title=f"J-space patch steering — target gap "
        f"(layer {int(jlens_out.jlens_source_layer)}, basis {jlens_out.jlens_basis}, scale {JLENS_PATCH_SCALE})",
    )
    print(
        f"feature-mediated delta {fm_post_gap - fm_pre_gap:+.3f} (scale {INTERVENTION_SCALE_FACTOR}) | "
        f"direct-hook add delta {direct_post_gap - direct_pre_gap:+.3f} (scale {INTERVENTION_SCALE_FACTOR}) | "
        f"J-space patch delta {jl_post_gap - jl_pre_gap:+.3f} "
        f"(basis {jlens_out.jlens_basis}, scale {JLENS_PATCH_SCALE})"
    )
    assert jl_post_gap > jl_pre_gap, "J-space patch steering should push the gap toward the target concept"
    assert jl_post_gap > 0, "a scale-1.0 workspace swap at this layer should flip the answer outright"
else:
    print(f"[SKIPPED] J-space section: {jspace_skip_reason}")

pulled speediedan/jlens_steering_ops @ 872760c7ca


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,16.075%,25.1250,26.8750,+1.7500
Color,15.106%,1.164%,27.0000,24.2500,-2.7500
Gap (Fruit − Color),,,-1.8750,+2.6250,+4.5000


feature-mediated delta +9.375 (scale 20.0) | direct-hook add delta +2.000 (scale 20.0) | J-space patch delta +4.500 (basis jlens_norm_aware, scale 1.0)


## 4c. Attribution comparison: features vs J-lens directions

The feature path explains the gap change with SAE features (cell 3); the J-space path explains the
**same** change with J-lens directions. `interpretune`'s subspace attribution splits the first-order
prediction of the patch's effect per direction: with $V = [v_s\ v_t]$, the displacement $\Delta h$ the
swap applied, the gradient $g = \partial \mathrm{gap} / \partial h$ at the lens site, pseudoinverse
coordinates $\Delta c = V^{+}\Delta h$ (the read side `patch` mode uses) and per-direction readouts
$w = V^{\top} g$, direction $i$ explains

$$
a_i = w_i \Delta c_i, \qquad \sum_i a_i + r = g^{\top}\Delta h,
$$

so the shares plus the remainder $r$ reconstruct the prediction exactly, and a dictionary that
explains nothing reports a large remainder rather than large shares. Here the dictionary is the pair
that produced the displacement, so $r$ is numerically zero and the interesting numbers are the split
between the poles and the gap between the linear prediction and the measured change (everything
nonlinear downstream of layer 24).

The cell takes one forward and backward pass on the underlying model for $h$ and $g$ (a forward hook at
the lens site, autograd on the gap), reuses the shared `apply_intervention` maths for $\Delta h$, and cross-checks each readout against a central
difference of the gap through `add`-mode probes along the same direction (the two agree to the
extent the gap is locally linear). The feature column is cell 3's signed influences, top five by
magnitude. Both columns report fractions within their own vocabulary: gap units per coordinate and
row-normalized graph influence are not comparable as raw numbers.


In [9]:
# @title 4c: Attribution comparison — features vs J-lens directions { display-mode: "form" }
# One forward + backward on the underlying model gives the clean activation h at the lens site and the
# gradient g of the gap there; the shared apply_intervention maths gives the swap's displacement; the
# tested subspace_attribution_scores splits the first-order prediction per pole. Central-difference
# slopes through add-mode probes cross-check the readouts. Same logit change, two vocabularies.
from interpretune.analysis.backends import InterventionSpec, apply_intervention
from interpretune.analysis.optools import jlens_direction_rows, resolve_jlens_layer, resolve_unembed_and_norm_scale
from interpretune.analysis.ops.bundled.jlens.jlens_ops import subspace_attribution_scores
from it_examples.utils.nb_ui_utils import display_attribution_comparison

if jspace_ran:
    _info = resolve_unembed_and_norm_scale(module)
    # MODEL_NAME coincides with the lens repository's model id on this demo substrate
    _jj, _jlayer, _ = resolve_jlens_layer(module, {}, {"jlens_model_id": MODEL_NAME, "jlens_layer": JLENS_LAYER})
    _V = jlens_direction_rows(_info, [target_a_id, target_b_id], _jj, apply_norm=True).detach().float()
    _site = f"blocks.{_jlayer}.hook_resid_post"
    _pole_labels = [f"{CONCEPT_TARGET_TOKENS[0]}-pole", f"{CONCEPT_TARGET_TOKENS[1]}-pole"]

    # h and g = d(gap)/dh at the site, from one forward + backward on the underlying model. The backend's
    # gradient seam saves only SAE-spliced sub-hooks, so a bare residual site is read through a forward hook on
    # the module that produces it (the HF decoder layer under an nnsight model, the TL hook point otherwise),
    # with the captured activation made a graph leaf so autograd reaches it whatever the parameters' grad state.
    # nnsight wraps the HF module (as `_model`, `_module` on older releases); TL-shaped models carry `blocks`.
    _hf = getattr(module.model, "_model", None) or getattr(module.model, "_module", None)
    if _hf is not None:
        _target = _hf.model.layers[_jlayer]
        _forward = lambda: _hf(input_ids=batch["input_ids"], attention_mask=batch.get("attention_mask")).logits
        _ids = batch["input_ids"]
    else:
        _target = module.model.blocks[_jlayer].hook_resid_post
        _forward = lambda: module.model(batch["input"])
        _ids = batch["input"]
    _cache: dict[str, torch.Tensor] = {}

    def _capture(_mod, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        leaf = hidden.detach().requires_grad_(True)
        _cache["h"] = leaf
        return (leaf,) + tuple(output[1:]) if isinstance(output, tuple) else leaf

    _handle = _target.register_forward_hook(_capture)
    try:
        with torch.enable_grad():
            _logits = _forward()
            _metric = _logits[0, -1, target_a_id] - _logits[0, -1, target_b_id]
            (_grad,) = torch.autograd.grad(_metric, _cache["h"])
    finally:
        _handle.remove()
    _h = _cache["h"].detach().float().cpu()
    _grad = _grad.detach().float().cpu()
    _last = int(_ids.shape[1]) - 1

    # The swap's displacement at the last token, from the same maths 4b's patch ran.
    _spec = InterventionSpec(intervention_tensor=_V.cpu(), mode="patch", scale_factor=JLENS_PATCH_SCALE)
    _delta_h = (apply_intervention(_h.clone(), _spec, last_pos=_last) - _h)[0, _last]
    _attr = subspace_attribution_scores(
        _grad[0, _last], _delta_h, _V.cpu(), [target_a_id, target_b_id], jlens_out.jlens_basis
    )

    def _gap_after_add(tensor, scale):
        module.analysis_cfg = AnalysisCfg(target_op=it.model_fwd_intervention, ignore_manual=True, save_tokens=False)
        init_analysis_cfgs(module, [module.analysis_cfg])
        _probe = AnalysisBatch(
            prompts=[prompt],
            logit_target_ids=torch.tensor([target_a_id], dtype=torch.long),
            concept_group_a_token_ids=[target_a_id],
            concept_group_b_token_ids=[target_b_id],
            intervention_hook_pattern=_site,
            intervention_mode="add",
            intervention_tensor=tensor,
            intervention_scale_factor=scale,
        )
        _out = it.model_fwd_intervention(module, _probe, batch, 0)
        _logits = _out.post_intervention_logits.float().cpu().reshape(-1)
        return float(_logits[target_a_id] - _logits[target_b_id])

    _EPS = 1.0
    _slopes = [(_gap_after_add(_V[i], _EPS) - _gap_after_add(_V[i], -_EPS)) / (2 * _EPS) for i in range(2)]

    _cmp = display_attribution_comparison(
        _attr,
        _pole_labels,
        finite_difference_slopes=_slopes,
        measured_delta=jl_post_gap - jl_pre_gap,
        features=steered_features,
        feature_scores=pipeline_results.top_feature_scores.tolist(),
        feature_explanations=feature_explanations,
        neuronpedia_model=NEURONPEDIA_MODEL_ID,
        neuronpedia_set=NEURONPEDIA_SOURCE_SET,
        neuronpedia_base_url=DASHBOARD_BASE_URL,
        top_n=5,
        title=(
            f"Attribution comparison — J-lens pair (layer {_jlayer}, basis {_attr['basis']}) "
            f"vs top SAE features (signed influence)"
        ),
    )
    print(
        f"first-order prediction {_cmp.predicted_delta:+.3f} = explained {_cmp.attribution_total:+.3f} + remainder "
        f"{_cmp.unexplained_remainder:+.3f}; measured patch delta {jl_post_gap - jl_pre_gap:+.3f} | pole fractions "
        + ", ".join(f"{lbl} {frac:.1%}" for lbl, frac in zip(_cmp.direction_labels, _cmp.direction_fractions))
    )
    # The pair produced the displacement, so the pair explains it: a remainder that is not numerically
    # zero means the dictionary and the displacement disagree about the basis, which is a defect.
    assert abs(_cmp.unexplained_remainder) < 0.01 * abs(_cmp.predicted_delta), "pair should explain its own patch"
else:
    print(f"[SKIPPED] attribution comparison: {jspace_skip_reason}")

first-order prediction +4.168 = explained +4.168 + remainder +0.000; measured patch delta +4.500 | pole fractions Fruit-pole 60.1%, Color-pole 39.9%


## 5. Input/output decoupling analysis (graph hydration + UMAP)

Attribution-selected steering features are chosen for their *output* effect (decoder projection
onto the target logit difference), while dashboard explanations describe their *input* behavior
(the contexts they fire on) — the two can decouple sharply. A recurring special case is the
**suppressor-motif exemplar**: a feature that *fires on concept contexts yet projects against the
concept token* (redundancy suppression under next-token training) — causally ideal for sign-aware
steering, semantically confusing on its dashboard. This step demonstrates those mechanics
directly, and in doing so demos the framework's **graph hydration** capability
(`analysis_backend.hydrate_graph_from_batch`) for detailed post-hoc analysis of a persisted
attribution result:

1. hydrate the step-2 attribution graph; highlight the prompt's concept-token positions;
2. compute each analyzed feature's **input profile** (activation mass at concept positions) and
   **output profile** (signed decoder projection onto the unit target-token unembed difference)
   via the shared `feature_io_profiles` helper;
3. render the decoupling table (signature column flags `decoupled` and `suppressor-motif` rows);
4. project decoder vectors to 2D (UMAP, PCA fallback — tooling shared with the latent-dynamics
   notebooks) with hover details per analyzed feature. Axis tick numbers are intentionally hidden:
   UMAP coordinates are non-metric (arbitrary rotation/scale; only local neighborhood structure is
   meaningful).

Expect roughly 1 of the 5 attribution picks to be input-aligned — a fruit-context feature that is
also a suppressor-motif exemplar (historically L25/16131, with all-`Fruit` negative logits) — and
the rest to be decoupled output machinery.


In [10]:
# @title 5: Decoupling analysis via hydrated graph + UMAP { display-mode: "form" }
import numpy as np

from interpretune.analysis.backends import require_analysis_backend
from it_examples.utils.example_helpers import concept_token_positions, feature_io_profiles
from it_examples.utils.nb_ui_utils import (
    display_concept_positions,
    display_feature_decoupling_table,
    plot_decoder_projection_map,
)

analysis_backend = require_analysis_backend(module)
graph = analysis_backend.hydrate_graph_from_batch(pipeline_results)

# concept-token positions derived from the demo's own concept groups + probe/target tokens
concept_words = {w.lower() for w in (*fruits, *colors, *CONCEPT_TARGET_TOKENS, "orange")}
prompt_token_ids = [int(t) for t in graph.input_tokens]
concept_positions = concept_token_positions(tokenizer, prompt_token_ids, sorted(concept_words))
display_concept_positions(tokenizer, prompt_token_ids, concept_positions)

embed_weight = analysis_backend.get_embedding_weight(module).detach().float().cpu()
target_diff = embed_weight[target_a_id] - embed_weight[target_b_id]
target_diff = target_diff / target_diff.norm()
target_label = f"{CONCEPT_TARGET_TOKENS[0]}-{CONCEPT_TARGET_TOKENS[1]}"

analyzed_pairs = list(dict.fromkeys((int(f[0]), int(f[-1])) for f in steered_features))
all_explanations = dict(feature_explanations)

transcoder_set = getattr(module.replacement_model.transcoders, "_module", module.replacement_model.transcoders)
profiles = feature_io_profiles(graph, analyzed_pairs, target_diff, transcoder_set, concept_positions)
display_feature_decoupling_table(profiles, all_explanations, target_label=target_label)

# decoder vectors for the interactive projection map (analyzed + random active-feature background)
rng = np.random.default_rng(17)
active_rows = graph.active_features.cpu()
background_pool = sorted({(int(r[0]), int(r[2])) for r in active_rows} - set(analyzed_pairs))
background_idx = rng.choice(len(background_pool), size=min(300, len(background_pool)), replace=False)
background_pairs = [background_pool[i] for i in background_idx]


def _decoder_rows(pairs):
    return torch.stack(
        [transcoder_set._get_decoder_vectors(lyr, torch.tensor([ft]))[0].detach().float().cpu() for lyr, ft in pairs]
    )


plot_decoder_projection_map(
    profiles,
    _decoder_rows(analyzed_pairs),
    _decoder_rows(background_pairs),
    feature_explanations=all_explanations,
    target_label=target_label,
    title="Steering-feature decoder map",
)

Feature,Input concept share,Act mass,Output proj (Fruit-Color),Signature,Explanation
L25/16131,0.481,346.72,-0.2961,suppressor-motif,"the comparison of two varieties of fruit, relating to size, taste, color, and ge"
L24/3865,0.097,191.47,+0.0264,,"dollar signs and other currency symbols, potentially alongside numbers or relate"
L25/13210,0.000,33.75,+0.0227,,grammatical structures and parts of speech like noun phrases and verb phrases
L24/5999,0.347,923.50,+0.0121,,"language related to institutions, negative situations, the internet, and program"
L24/13277,0.186,1297.00,+0.0061,,words related to questions and requests


## Summary

- Feature-mediated and direct-hook steering paths on one proven example, one backend per run —
  both driven by the same embed-basis concept direction (store-basis directions are a
  `tests/nb_experiments` research thread, see `EXPERIMENT_STATUS.md`).
- Semantic grounding via public neuronpedia.org feature dashboards and explanations.
- Input/output decoupling mechanics demonstrated via graph hydration + decoder-space UMAP,
  doubling as a demo of persisted-graph post-hoc analysis.
- Local Neuronpedia dashboards, locally generated explanations, and user-curated feature steering:
  [`ct_concept_steering_demo_local_np.ipynb`](ct_concept_steering_demo_local_np.ipynb).
- Deeper coverage of the underlying op pipeline (per-op invocation, native + hub composition):
  see `ct_analysis_backend_demo.ipynb`. Capability map:
  `tests/nb_experiments/intervention_capabilities_overview.md`.
